# Web Scraping Exercises\n\nThis Google Colab notebook contains all six exercises using `urlopen()` and `BeautifulSoup`.

## Setup

In [ ]:
!pip -q install beautifulsoup4

In [ ]:
import random\nimport re\nfrom datetime import datetime\nfrom urllib.parse import quote, urlencode\nfrom urllib.request import Request, urlopen\n\nfrom bs4 import BeautifulSoup\n\n\ndef fetch_html(url):\n    request = Request(url, headers={\"User-Agent\": \"Mozilla/5.0\"})\n    with urlopen(request, timeout=20) as response:\n        return response.read().decode(\"utf-8\", errors=\"ignore\")

## Exercise 1: Parsing HTML With BeautifulSoup

In [ ]:
sports_html = \"\"\"\n<!DOCTYPE html>\n<html lang='en'>\n<head>\n    <meta charset='UTF-8'>\n    <meta name='viewport' content='width=device-width, initial-scale=1.0'>\n    <title>Sports World</title>\n</head>\n<body>\n    <header>\n        <h1>Welcome to Sports World</h1>\n        <p>Your one-stop destination for the latest sports news and videos.</p>\n    </header>\n\n    <nav>\n        <a href='#football'>Football</a>\n        <a href='#basketball'>Basketball</a>\n        <a href='#tennis'>Tennis</a>\n    </nav>\n\n    <section id='football'>\n        <h2>Football</h2>\n        <article>\n            <h3>Latest Football News</h3>\n            <p>Read about the latest football matches and player news.</p>\n            <div class='video'>\n                <iframe width='560' height='315' src='https://www.youtube.com/embed/football-video-id'></iframe>\n            </div>\n        </article>\n    </section>\n\n    <section id='basketball'>\n        <h2>Basketball</h2>\n        <article>\n            <h3>NBA Highlights</h3>\n            <p>Watch highlights from the latest NBA games.</p>\n            <div class='video'>\n                <iframe width='560' height='315' src='https://www.youtube.com/embed/basketball-video-id'></iframe>\n            </div>\n        </article>\n    </section>\n\n    <section id='tennis'>\n        <h2>Tennis</h2>\n        <article>\n            <h3>Grand Slam Updates</h3>\n            <p>Get the latest updates from the world of Grand Slam tennis.</p>\n            <div class='video'>\n                <iframe width='560' height='315' src='https://www.youtube.com/embed/tennis-video-id'></iframe>\n            </div>\n        </article>\n    </section>\n</body>\n</html>\n\"\"\"\n\nsports_url = \"data:text/html;charset=utf-8,\" + quote(sports_html)\nwith urlopen(sports_url) as response:\n    html_content = response.read().decode(\"utf-8\")\n\nsoup = BeautifulSoup(html_content, \"html.parser\")\n\ntitle = soup.title.get_text(strip=True) if soup.title else \"No title found\"\nparagraphs = [p.get_text(strip=True) for p in soup.find_all(\"p\")]\nlinks = [a.get(\"href\") for a in soup.find_all(\"a\") if a.get(\"href\")]\n\nprint(\"Title:\", title)\nprint(\"\\nParagraphs:\")\nfor paragraph in paragraphs:\n    print(\"-\", paragraph)\n\nprint(\"\\nLinks:\")\nfor link in links:\n    print(\"-\", link)

## Exercise 2: Scraping robots.txt From Wikipedia

In [ ]:
robots_url = \"https://en.wikipedia.org/robots.txt\"\n\ntry:\n    robots_content = fetch_html(robots_url)\n    print(robots_content)\nexcept Exception as error:\n    print(\"Could not download robots.txt:\", error)

## Exercise 3: Extracting Headers From Wikipedia's Main Page

In [ ]:
wikipedia_url = \"https://en.wikipedia.org/wiki/Main_Page\"\n\ntry:\n    html = fetch_html(wikipedia_url)\n    soup = BeautifulSoup(html, \"html.parser\")\n    headers = soup.find_all([\"h1\", \"h2\", \"h3\", \"h4\", \"h5\", \"h6\"])\n\n    for header in headers:\n        text = header.get_text(\" \", strip=True)\n        if text:\n            print(f\"{header.name}: {text}\")\nexcept Exception as error:\n    print(\"Could not extract Wikipedia headers:\", error)

## Exercise 4: Checking for Page Title

In [ ]:
def page_has_title(url):\n    html = fetch_html(url)\n    soup = BeautifulSoup(html, \"html.parser\")\n    return soup.title is not None and soup.title.get_text(strip=True) != \"\"\n\n\ntry:\n    if page_has_title(\"https://en.wikipedia.org/wiki/Main_Page\"):\n        print(\"The page contains a title.\")\n    else:\n        print(\"The page does not contain a title.\")\nexcept Exception as error:\n    print(\"Could not check the page title:\", error)

## Exercise 5: Analyzing US-CERT Security Alerts

In [ ]:
def get_cisa_alert_count_for_current_year(max_pages=10):\n    current_year = datetime.now().year\n    alert_count = 0\n    seen_links = set()\n    base_url = \"https://www.cisa.gov/news-events/cybersecurity-advisories\"\n\n    for page in range(max_pages):\n        params = urlencode({\"f[0]\": \"advisory_type:93\", \"page\": page})\n        url = f\"{base_url}?{params}\"\n        html = fetch_html(url)\n        soup = BeautifulSoup(html, \"html.parser\")\n\n        page_links = [link for link in soup.find_all(\"a\", href=True) if \"/news-events/alerts/\" in link[\"href\"]]\n        new_links_found = False\n\n        for link in page_links:\n            href = link[\"href\"]\n            if href in seen_links:\n                continue\n\n            seen_links.add(href)\n            new_links_found = True\n\n            card_text = link.parent.get_text(\" \", strip=True) if link.parent else link.get_text(\" \", strip=True)\n            if str(current_year) in card_text or re.search(rf\"\\b{current_year}\\b\", href):\n                alert_count += 1\n\n        if not new_links_found:\n            break\n\n    return current_year, alert_count\n\n\ntry:\n    year, count = get_cisa_alert_count_for_current_year()\n    print(f\"Number of US-CERT security alerts issued in {year}: {count}\")\nexcept Exception as error:\n    print(\"Could not count US-CERT security alerts:\", error)

## Exercise 6: Scraping Movie Details

In [ ]:
def get_imdb_movies():\n    url = \"https://www.imdb.com/list/ls091294718/\"\n    html = fetch_html(url)\n    soup = BeautifulSoup(html, \"html.parser\")\n    movies = []\n\n    old_style_cards = soup.select(\".lister-item\")\n    for card in old_style_cards:\n        title_tag = card.select_one(\".lister-item-header a\")\n        year_tag = card.select_one(\".lister-item-year\")\n        summary_tag = card.select_one(\"p.text-muted:not(.text-small)\")\n\n        if title_tag:\n            movies.append({\n                \"name\": title_tag.get_text(strip=True),\n                \"year\": year_tag.get_text(strip=True) if year_tag else \"Unknown year\",\n                \"summary\": summary_tag.get_text(\" \", strip=True) if summary_tag else \"No summary found.\",\n            })\n\n    if movies:\n        return movies\n\n    modern_cards = soup.select(\"li.ipc-metadata-list-summary-item\")\n    for card in modern_cards:\n        title_tag = card.select_one(\"h3\")\n        summary_tag = card.select_one(\".ipc-html-content-inner-div\")\n        card_text = card.get_text(\" \", strip=True)\n        year_match = re.search(r\"\\b(19|20)\\d{2}\\b\", card_text)\n\n        if title_tag:\n            title = re.sub(r\"^\\d+\\.\\s*\", \"\", title_tag.get_text(strip=True))\n            movies.append({\n                \"name\": title,\n                \"year\": year_match.group(0) if year_match else \"Unknown year\",\n                \"summary\": summary_tag.get_text(\" \", strip=True) if summary_tag else \"No summary found.\",\n            })\n\n    return movies\n\n\ntry:\n    movies = get_imdb_movies()\n\n    if len(movies) >= 10:\n        selected_movies = random.sample(movies, 10)\n    else:\n        selected_movies = movies\n\n    if not selected_movies:\n        print(\"No movies found. IMDb may have blocked the request or changed the page structure.\")\n    else:\n        for index, movie in enumerate(selected_movies, start=1):\n            print(f\"{index}. {movie['name']} - {movie['year']}\")\n            print(movie[\"summary\"])\n            print()\nexcept Exception as error:\n    print(\"Could not scrape IMDb movies:\", error)